# 🧪 Step 1: Research & Data Modelling  
## Petrol Stations & EV Charging Points in Berlin

**PR Branch Name:** `petrol-stations-berlin-data-modelling`

This notebook documents **Step 1** of the Gas / Petrol Stations data layer
for Berlin, following the same structure as the *Supermarkets* layer.

---

### Scope
- Petrol / fuel stations
- EV charging stations.

---

### Objectives
- Identify and document data sources
- Fetch raw POI data from OpenStreetMap
- Select relevant columns for modelling
- Draft a standardized schema
- Plan transformation and enrichment steps



## 1.1 Data Source Discovery

### Topic
Petrol Stations & EV Charging Infrastructure in Berlin

### Main Data Source
**OpenStreetMap (OSM)**  
- Public, crowdsourced geospatial database  
- Update frequency: Continuous (near real-time)  
- Data type: Dynamic (API-based)

### Relevant OSM Tags
- `amenity=fuel` → Petrol / gas stations
- `amenity=charging_station` → EV charging points

### Reason for Selection
- Broad coverage of petrol stations across Berlin
- Includes coordinates, names, operators, opening hours, and amenities
- Open license and easy to query programmatically

### Optional / Enrichment Sources
- **Berlin Open Data Portal**  
  - Official EV charging datasets
  - Static or semi-static

- **Wikidata**
  - Operator / brand enrichment



In [1]:
import sys
print(sys.executable)


/opt/anaconda3/bin/python


In [2]:
import sys
print("Python:", sys.executable)
print("Version:", sys.version)


Python: /opt/anaconda3/bin/python
Version: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:11:29) [Clang 20.1.8 ]


In [3]:
import osmnx as ox
import geopandas as gpd
import pandas as pd

print("✅ imports working")


✅ imports working


In [4]:
ox.settings.use_cache = True
ox.settings.log_console = True


## 📍 Fetch Petrol Stations & EV Charging Points from OpenStreetMap

In this section, we query OpenStreetMap using **OSMnx** to retrieve:

- Petrol stations (`amenity=fuel`)
- EV charging stations (`amenity=charging_station`)

The result is a GeoDataFrame containing geometry and OSM metadata.


In [5]:
tags_fuel = {"amenity": "fuel"}
gdf_fuel = ox.features.features_from_place("Berlin, Germany", tags=tags_fuel)

print("Fuel stations fetched:", len(gdf_fuel))
gdf_fuel.head(3)


Fuel stations fetched: 289


geometry amenity brand brand:wikidata  \
element id                                                                 
node    16541597   POINT (13.3456 52.54631)    fuel  Aral        Q565734   
        26756830  POINT (13.32822 52.53069)    fuel  Aral        Q565734   
        26867411   POINT (13.2945 52.50197)    fuel   NaN            NaN   

                 brand:wikipedia compressed_air fuel:HGV_diesel fuel:diesel  \
element id                                                                    
node    16541597      en:Aral AG            yes          retail         yes   
        26756830         de:Aral            yes             NaN         yes   
        26867411             NaN            yes             NaN         yes   

                 fuel:e10 fuel:octane_102  ... landuse  \
element id                                 ...           
node    16541597      yes          retail  ...     NaN   
        26756830      yes             yes  ...     NaN   
        26867411      yes             NaN  ...     NaN   

                 service:vehicle:car_repair service:vehicle:inspection  \
element id                                                               
node    16541597                        NaN                        NaN   
        26756830                        NaN                        NaN   
        26867411                        NaN                        NaN   

                 fuel:octane_92 height wikimedia_commons payment:H2_Delivery  \
element id                                                                     
node    16541597            NaN    NaN               NaN                 NaN   
        26756830            NaN    NaN               NaN                 NaN   
        26867411            NaN    NaN               NaN                 NaN   

                 payment:HyLane payment:ryd_pay start_date:fuel:h35  
element id                                                           
node    16541597            NaN             NaN                 NaN  
        26756830            NaN             NaN                 NaN  
        26867411            NaN             NaN                 NaN  

[3 rows x 145 columns]

In [6]:
tags_ev = {"amenity": "charging_station"}
gdf_ev = ox.features.features_from_place("Berlin, Germany", tags=tags_ev)

print("Charging stations fetched:", len(gdf_ev))
gdf_ev.head(3)


Charging stations fetched: 1563


geometry           amenity amperage  \
element id                                                                
node    425089052   POINT (13.3227 52.51369)  charging_station       32   
        429740871  POINT (13.38087 52.52366)  charging_station       32   
        429951629  POINT (13.32903 52.49941)  charging_station       32   

                  authentication:short_message bicycle capacity  \
element id                                                        
node    425089052                          yes      no        2   
        429740871                          yes     NaN        2   
        429951629                          yes     NaN        2   

                        operator                    ref socket:type2  \
element id                                                             
node    425089052  RWE-Effizienz  BA-0602-4 / BA-0603-0            2   
        429740871           E.ON  BA-0500-2 / BA-0501-9            2   
        429951629         Innogy  BA-0798-1 / BA-0707-5            2   

                                         source  ... socket:unknown  \
element id                                       ...                  
node    425089052  RWE Effizienz GmbH, Dortmund  ...            NaN   
        429740871                        survey  ...            NaN   
        429951629  RWE Effizienz GmbH, Dortmund  ...            NaN   

                  socket:unknown:output markings restriction street:name  \
element id                                                                 
node    425089052                   NaN      NaN         NaN         NaN   
        429740871                   NaN      NaN         NaN         NaN   
        429951629                   NaN      NaN         NaN         NaN   

                  capacity:disabled surface ref:goingelectric  \
element id                                                      
node    425089052               NaN     NaN               NaN   
        429740871               NaN     NaN               NaN   
        429951629               NaN     NaN               NaN   

                  socket:chademo:current socket:chademo:voltage  
element id                                                       
node    425089052                    NaN                    NaN  
        429740871                    NaN                    NaN  
        429951629                    NaN                    NaN  

[3 rows x 182 columns]

## Combine Petrol & EV Charging Data

Both datasets are combined into a single POI layer.
A new column `poi_type` is added to distinguish:

- `fuel_station`
- `charging_station`


In [7]:
gdf_fuel["poi_type"] = "fuel_station"
gdf_ev["poi_type"] = "charging_station"

gdf = pd.concat([gdf_fuel, gdf_ev], axis=0)
gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs=gdf_fuel.crs)

print("Total POIs:", len(gdf))
gdf.head(3)


Total POIs: 1852


geometry amenity brand brand:wikidata  \
element id                                                                 
node    16541597   POINT (13.3456 52.54631)    fuel  Aral        Q565734   
        26756830  POINT (13.32822 52.53069)    fuel  Aral        Q565734   
        26867411   POINT (13.2945 52.50197)    fuel   NaN            NaN   

                 brand:wikipedia compressed_air fuel:HGV_diesel fuel:diesel  \
element id                                                                    
node    16541597      en:Aral AG            yes          retail         yes   
        26756830         de:Aral            yes             NaN         yes   
        26867411             NaN            yes             NaN         yes   

                 fuel:e10 fuel:octane_102  ... taxi socket:unknown  \
element id                                 ...                       
node    16541597      yes          retail  ...  NaN            NaN   
        26756830      yes             yes  ...  NaN            NaN   
        26867411      yes             NaN  ...  NaN            NaN   

                 socket:unknown:output markings restriction street:name  \
element id                                                                
node    16541597                   NaN      NaN         NaN         NaN   
        26756830                   NaN      NaN         NaN         NaN   
        26867411                   NaN      NaN         NaN         NaN   

                 capacity:disabled ref:goingelectric socket:chademo:current  \
element id                                                                    
node    16541597               NaN               NaN                    NaN   
        26756830               NaN               NaN                    NaN   
        26867411               NaN               NaN                    NaN   

                 socket:chademo:voltage  
element id                               
node    16541597                    NaN  
        26756830                    NaN  
        26867411                    NaN  

[3 rows x 278 columns]

## 1.2 Modelling & Planning

### Selected Key Columns
- station_name
- street, housenumber, postcode, city
- brand / operator
- opening_hours
- contact info (phone, email, website)
- petrol vs charging type
- latitude, longitude
- district, neighborhood (derived later)
- data_source

### Known Data Issues
- Address fields may be partially missing
- Fuel types are inconsistently tagged
- Contact information often incomplete

### Planned Transformations
- Rename columns to a standardized format
- Extract latitude and longitude from geometry
- Enrich with district & neighborhood via spatial join
- Deduplicate overlapping POIs


In [8]:
columns = [
    "name",
    "addr:street", "addr:housenumber", "addr:postcode", "addr:city",
    "opening_hours",
    "brand", "operator",
    "phone", "contact:email", "website",
    "geometry",
    "poi_type"
]

gdf_station = gdf[[c for c in columns if c in gdf.columns]].copy()
print("Filtered dataset shape:", gdf_station.shape)
gdf_station.head(3)


Filtered dataset shape: (1852, 13)


name         addr:street addr:housenumber  \
element id                                                              
node    16541597            Aral                 NaN              NaN   
        26756830            Aral       Beusselstraße               55   
        26867411  Bavaria petrol  Heilbronner Straße               14   

                 addr:postcode addr:city                      opening_hours  \
element id                                                                    
node    16541597           NaN       NaN                               24/7   
        26756830         10553    Berlin                               24/7   
        26867411         10711    Berlin  Mo-Fr 07:00-20:00; Sa 07:00-18:00   

                 brand        operator           phone contact:email  \
element id                                                             
node    16541597  Aral             NaN             NaN           NaN   
        26756830  Aral      Mike Seitz  +49 30 3914404           NaN   
        26867411   NaN  Bavaria Petrol             NaN           NaN   

                                                            website  \
element id                                                            
node    16541597                                                NaN   
        26756830  https://tankstelle.aral.de/berlin/beusselstras...   
        26867411                                                NaN   

                                   geometry      poi_type  
element id                                                 
node    16541597   POINT (13.3456 52.54631)  fuel_station  
        26756830  POINT (13.32822 52.53069)  fuel_station  
        26867411   POINT (13.2945 52.50197)  fuel_station

## Geometry Normalization & Coordinate Extraction

All geometries are normalized to `Point` objects.
Latitude and longitude are extracted for mapping and spatial joins.


In [9]:
# Ensure geometry is Point
gdf_station["geometry"] = gdf_station["geometry"].apply(
    lambda geom: geom if geom.geom_type == "Point" else geom.representative_point()
)

# Extract latitude and longitude
gdf_station["latitude"] = gdf_station.geometry.y
gdf_station["longitude"] = gdf_station.geometry.x


## Column Normalization

OSM-specific column names are renamed for clarity.
A `data_source` column is added for traceability.


In [10]:
gdf_station = gdf_station.rename(columns={
    "name": "station_name",
    "addr:street": "street",
    "addr:housenumber": "housenumber",
    "addr:postcode": "postcode",
    "addr:city": "city",
    "contact:email": "email"
})

gdf_station["data_source"] = "OSM"
gdf_station.head(3)


station_name              street housenumber postcode  \
element id                                                                  
node    16541597            Aral                 NaN         NaN      NaN   
        26756830            Aral       Beusselstraße          55    10553   
        26867411  Bavaria petrol  Heilbronner Straße          14    10711   

                    city                      opening_hours brand  \
element id                                                          
node    16541597     NaN                               24/7  Aral   
        26756830  Berlin                               24/7  Aral   
        26867411  Berlin  Mo-Fr 07:00-20:00; Sa 07:00-18:00   NaN   

                        operator           phone email  \
element id                                               
node    16541597             NaN             NaN   NaN   
        26756830      Mike Seitz  +49 30 3914404   NaN   
        26867411  Bavaria Petrol             NaN   NaN   

                                                            website  \
element id                                                            
node    16541597                                                NaN   
        26756830  https://tankstelle.aral.de/berlin/beusselstras...   
        26867411                                                NaN   

                                   geometry      poi_type   latitude  \
element id                                                             
node    16541597   POINT (13.3456 52.54631)  fuel_station  52.546315   
        26756830  POINT (13.32822 52.53069)  fuel_station  52.530694   
        26867411   POINT (13.2945 52.50197)  fuel_station  52.501974   

                  longitude data_source  
element id                               
node    16541597  13.345599         OSM  
        26756830  13.328225         OSM  
        26867411  13.294496         OSM

## Next Steps (to be done next)

- Data quality analysis (missing values, distributions)
- Draft standardized schema (`gas_stations_table`)
- Save raw data into `/sources`
- Create `/sources/README.md`
- Commit & open PR


In [11]:
[g for g in globals().keys() if g.startswith("gdf")]


['gdf_fuel', 'gdf_ev', 'gdf', 'gdf_station']

In [12]:
print("gdf_fuel:", len(gdf_fuel))
print("gdf_ev:", len(gdf_ev))
print("gdf:", len(gdf))
print("gdf_station:", len(gdf_station))


gdf_fuel: 289
gdf_ev: 1563
gdf: 1852
gdf_station: 1852


In [13]:
##Export RAW files
from pathlib import Path

SOURCES_DIR = Path("./sources")
SOURCES_DIR.mkdir(parents=True, exist_ok=True)

raw_geojson_path = SOURCES_DIR / "gas_stations_raw.geojson"
raw_csv_path = SOURCES_DIR / "gas_stations_raw.csv"

gdf.to_file(raw_geojson_path, driver="GeoJSON")
gdf.drop(columns="geometry").to_csv(raw_csv_path, index=False)

print("✅ Saved:")
print(" -", raw_geojson_path)
print(" -", raw_csv_path)


✅ Saved:
 - sources/gas_stations_raw.geojson
 - sources/gas_stations_raw.csv
